# Multidimensional IRT — exploratory analysis

Question: does the benchmark suite span more than one latent capability axis?
The canonical IRT model fits a single $C_m$ per model. The residual-correlation
diagnostic (`3_diagnostics/residual_corr.py`) hints at structure beyond 1D but cannot
estimate dimensionality.

This notebook fits **PPCA + ARD** on the (models × benchmarks) score matrix to get
a soft estimate of the effective rank, and inspects loadings for the axes that
survive ARD shrinkage. Scope: exploratory dimensionality + axis interpretation —
**not** a full multidimensional-IRT rebuild.

We use the full benchmark coverage (`include_all_benchmarks=True`), which re-injects
the 8 curated exclusions (ARC-AGI variants, HellaSwag, OpenBookQA, PIQA, SimpleBench,
VPCT, WinoGrande) from the pipeline's pre-exclusion intermediate. This gives PPCA
the broadest possible view of latent structure.


## Step 1 — Load the data

Uses `load_eci_data(include_all_benchmarks=True)` so the 8 curated exclusions
(ARC-AGI, ARC-AGI-2, HellaSwag, OpenBookQA, PIQA, SimpleBench, VPCT, WinoGrande)
are re-injected from `1_data/1_pipeline/intermediate/04_scores_deduped.csv`. This gives
the PPCA model the broadest possible benchmark coverage to look for latent
structure. We then
pivot the long-format observations into a `models × benchmarks` score
matrix with NaN for unobserved pairs — that's the input PPCA/ARD will
consume in step 2.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd

from data import load_eci_data

data = load_eci_data(include_all_benchmarks=True)
print(f"n_obs={data.n_obs}, n_models={data.n_models}, n_benchmarks={data.n_benchmarks}")
print(f"humans fitted: {int(data.is_human.sum())}")
print(f"zero scores:   {int(data.zero_score_mask.sum())}")

n_obs=3610, n_models=709, n_benchmarks=70
humans fitted: 7
zero scores:   34


In [2]:
# Long-form observations → wide score matrix Y (models × benchmarks).
# Unobserved (model, benchmark) pairs are NaN. We do not impute here;
# the PPCA/ARD step will handle missingness explicitly.
models     = data.mlookup["model"].to_numpy()
benchmarks = data.blookup["benchmark"].to_numpy()

Y = np.full((data.n_models, data.n_benchmarks), np.nan)
Y[data.model_idx, data.bench_idx] = data.scores

print(f"score matrix: {Y.shape}")
print(f"observed cells: {np.isfinite(Y).sum()} / {Y.size} "
      f"({np.isfinite(Y).mean()*100:.1f}% dense)")
print(f"per-model obs:     min={data.n_obs_per_model.min()}, "
      f"median={int(np.median(data.n_obs_per_model))}, "
      f"max={data.n_obs_per_model.max()}")
obs_per_bench = np.isfinite(Y).sum(axis=0)
print(f"per-benchmark obs: min={obs_per_bench.min()}, "
      f"median={int(np.median(obs_per_bench))}, "
      f"max={obs_per_bench.max()}")

score matrix: (709, 70)
observed cells: 3585 / 49630 (7.2% dense)
per-model obs:     min=1, median=3, max=41
per-benchmark obs: min=6, median=38, max=176


In [3]:
# Quick sanity peek — top-10 most-observed benchmarks and a slice of Y.
bench_obs = (pd.DataFrame({"benchmark": benchmarks, "n_obs": obs_per_bench})
             .sort_values("n_obs", ascending=False))
bench_obs.head(10)

,benchmark,n_obs
19,GPQA Diamond,176
46,OTIS Mock AIME 2024-2025,149
68,WeirdML,143
2,ARC-AGI,142
38,MMLU,138
3,ARC-AGI-2,133
35,MATH Level 5,111
17,FrontierMath,102
24,GSM8K,94
56,SimpleBench,82


## Step 1b — Distribution of observations per model

Before filtering, look at the shape of `n_obs_per_model`. PPCA/ARD needs
each row (model) to contribute enough observed cells to constrain its
latent score — single-observation rows are uninformative and add row-noise
without identifying any latent direction. The histogram below shows where
a sensible `LOW_OBS_THRESHOLD` cut would land.

In [4]:
from config import LOW_OBS_THRESHOLD

n = data.n_obs_per_model
print(f"LOW_OBS_THRESHOLD (config) = {LOW_OBS_THRESHOLD}")
print(f"models total: {len(n)}")
print()

# Cumulative table — how many models survive each candidate threshold,
# and how many observations they collectively carry.
rows = []
for k in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]:
    keep = n >= k
    rows.append({
        "threshold (>= k obs)": k,
        "models kept":          int(keep.sum()),
        "models dropped":       int((~keep).sum()),
        "obs kept":             int(n[keep].sum()),
        "obs dropped":          int(n[~keep].sum()),
        "% obs kept":           round(100 * n[keep].sum() / n.sum(), 1),
    })
pd.DataFrame(rows)

LOW_OBS_THRESHOLD (config) = 4
models total: 709



,threshold (>= k obs),models kept,models dropped,obs kept,obs dropped,% obs kept
0,1,709,0,3610,0,100.0
1,2,481,228,3382,228,93.7
2,3,372,337,3164,446,87.6
3,4,295,414,2933,677,81.2
4,5,245,464,2733,877,75.7
5,6,211,498,2563,1047,71.0
6,8,139,570,2090,1520,57.9
7,10,97,612,1736,1874,48.1
8,12,72,637,1472,2138,40.8
9,15,55,654,1257,2353,34.8


In [5]:
import plotly.express as px

# Histogram of obs-per-model, capped at 32 for readability.
fig = px.histogram(
    x=n, nbins=int(n.max()),
    labels={"x": "observations per model", "y": "model count"},
    title=f"Observations per model (n={len(n)} models, "
          f"median={int(np.median(n))}, max={int(n.max())})",
)
fig.add_vline(x=LOW_OBS_THRESHOLD, line_dash="dash", line_color="red",
              annotation_text=f"LOW_OBS_THRESHOLD={LOW_OBS_THRESHOLD}")
fig.update_layout(bargap=0.05, height=400)
fig.show()

# Phase A — PPCA + ARD

## A.1 Preprocessing

The `prepare_matrix(k)` function applies the deterministic prep:

1. **Filter rows.** Keep AI models with `n_obs ≥ k`; **drop the 7 human rows**
   (different population — would load on a spurious axis).
2. **Filter columns.** Drop any benchmark left with `< MIN_BENCH_OBS` observations
   among the kept models — a column needs enough points for a stable mean and to
   contribute a real loading.
3. **Clip + logit.** Clip scores to `[ECI_EPS, 1 − ECI_EPS]` (`ECI_EPS = 1e-3`) —
   the same fixed-epsilon boundary handling the main model now uses (`model.py`;
   reverted from Smithson–Verkuilen) — then apply `logit`. This puts scores on the
   model's own latent scale, linearizes, and homogenizes variance so PPCA's
   Gaussian-noise assumption is reasonable.
4. **Column-center** each benchmark over its observed cells only (subtract the
   per-benchmark mean). After centering, no intercept term is needed in the model
   and PC1 won't just be "benchmark difficulty."

Returns the long-form observed vector + reindexed `(model_idx, bench_idx)` — the
same indexing pattern the main model uses, so the PPCA likelihood touches only
observed cells (no imputation).

In [6]:
from dataclasses import dataclass
from scipy.special import logit, expit

from config import ECI_EPS

MIN_BENCH_OBS = 3  # a benchmark needs at least this many points among kept models


@dataclass
class PreppedData:
    x: np.ndarray            # (n_obs,) logit-transformed, column-centered scores
    model_idx: np.ndarray    # (n_obs,) reindexed 0..M-1
    bench_idx: np.ndarray    # (n_obs,) reindexed 0..B-1
    M: int                   # kept models
    B: int                   # kept benchmarks
    model_names: np.ndarray  # (M,)
    bench_names: np.ndarray  # (B,)
    col_mean: np.ndarray     # (B,) per-benchmark mean removed (logit scale)
    n_obs: int
    k: int


def prepare_matrix(data, k: int, exclude_humans: bool = True,
                   min_bench_obs: int = MIN_BENCH_OBS) -> PreppedData:
    """Filter + clip + logit + column-center on the observed cells of `data`."""
    model_names_all = data.mlookup["model"].to_numpy()
    bench_names_all = data.blookup["benchmark"].to_numpy()

    # 1. Row filter: AI models with >= k obs, drop humans.
    keep_model = data.n_obs_per_model >= k
    if exclude_humans:
        keep_model &= ~data.is_human
    obs_keep = keep_model[data.model_idx]

    m_old = data.model_idx[obs_keep]
    b_old = data.bench_idx[obs_keep]
    y     = data.scores[obs_keep]

    # 2. Column filter: benchmarks with >= min_bench_obs among the kept rows.
    bench_counts = np.bincount(b_old, minlength=data.n_benchmarks)
    keep_bench = bench_counts >= min_bench_obs
    col_ok = keep_bench[b_old]
    m_old, b_old, y = m_old[col_ok], b_old[col_ok], y[col_ok]

    # Reindex surviving models / benchmarks to contiguous 0-based ids.
    kept_models = np.sort(np.unique(m_old))
    kept_bench  = np.sort(np.unique(b_old))
    m_remap = {old: new for new, old in enumerate(kept_models)}
    b_remap = {old: new for new, old in enumerate(kept_bench)}
    model_idx = np.array([m_remap[i] for i in m_old])
    bench_idx = np.array([b_remap[j] for j in b_old])
    M, B = len(kept_models), len(kept_bench)

    # 3. Fixed-epsilon boundary clip + logit. Matches the main model's current
    #    convention (model.py: np.clip(scores, ECI_EPS, 1 - ECI_EPS)); interior
    #    scores untouched, only exact-0/1 boundary rows are pulled inward.
    y_clip = np.clip(y, ECI_EPS, 1.0 - ECI_EPS)
    z = logit(y_clip)

    # 4. Column-center over observed cells only.
    col_mean = np.array([z[bench_idx == j].mean() for j in range(B)])
    x = z - col_mean[bench_idx]

    return PreppedData(
        x=x, model_idx=model_idx, bench_idx=bench_idx, M=M, B=B,
        model_names=model_names_all[kept_models],
        bench_names=bench_names_all[kept_bench],
        col_mean=col_mean, n_obs=len(y), k=k,
    )


prep4 = prepare_matrix(data, k=4)
print(f"k={prep4.k}: {prep4.M} models × {prep4.B} benchmarks, "
      f"{prep4.n_obs} observed cells "
      f"({100*prep4.n_obs/(prep4.M*prep4.B):.1f}% dense)")
dropped_bench = sorted(set(benchmarks) - set(prep4.bench_names))
print(f"benchmarks dropped (< {MIN_BENCH_OBS} obs after row filter): "
      f"{dropped_bench if dropped_bench else 'none'}")
print(f"centered-logit x: mean={prep4.x.mean():+.3f} (≈0 by construction), "
      f"sd={prep4.x.std():.3f}, range=[{prep4.x.min():.2f}, {prep4.x.max():.2f}]")

k=4: 291 models × 69 benchmarks, 2873 observed cells (14.3% dense)
benchmarks dropped (< 3 obs after row filter): ['SuperGLUE']
centered-logit x: mean=-0.000 (≈0 by construction), sd=1.149, range=[-6.56, 6.87]


In [7]:
import pymc as pm

K = 8  # overcomplete latent dim (> the 2-3 axes we expect) — ARD prunes the surplus


def build_ppca(prep: PreppedData, K: int = K, noise: str = "isotropic"):
    """PPCA + ARD model on the observed cells of `prep`.

    noise="isotropic" → single sigma (Phase A, PPCA).
    noise="diagonal"  → per-benchmark sigma_b (factor-analysis variant). Kept in the factory for future sensitivity checks; not exercised here.
    """
    coords = {
        "model": prep.model_names,
        "bench": prep.bench_names,
        "latent": np.arange(K),
    }
    with pm.Model(coords=coords) as model:
        mi = pm.Data("model_idx", prep.model_idx, dims="obs_id")
        bi = pm.Data("bench_idx", prep.bench_idx, dims="obs_id")

        # Z: each model's position on the K hidden axes — unit scale (pins Z/W split).
        Z = pm.Normal("Z", 0.0, 1.0, dims=("model", "latent"))

        # tau: per-axis ARD scale. Unused axes shrink toward ~0 (switched off);
        # needed axes grow. W: benchmark loadings, non-centered (W_z * tau).
        tau = pm.LogNormal("tau", mu=np.log(0.3), sigma=1.0, dims="latent")
        W_z = pm.Normal("W_z", 0.0, 1.0, dims=("bench", "latent"))
        W = pm.Deterministic("W", W_z * tau, dims=("bench", "latent"))

        # Predicted (centered-logit) score = <Z_m, W_b>, over observed cells only.
        mu = (Z[mi] * W[bi]).sum(axis=-1)

        if noise == "isotropic":
            sigma = pm.HalfNormal("sigma", 2.0)          # one shared noise SD (PPCA)
        elif noise == "diagonal":
            sigma_b = pm.HalfNormal("sigma_b", 2.0, dims="bench")  # per-benchmark
            sigma = sigma_b[bi]
        else:
            raise ValueError(noise)

        pm.Normal("obs", mu=mu, sigma=sigma, observed=prep.x, dims="obs_id")
    return model


## A.2 PPCA + ARD fit (K=8, 8 chains × 2000 draws)

Fit PPCA + ARD on the all-benchmarks matrix produced by `prepare_matrix(data, k=4)`.

8 chains × 2000 draws = 16,000 total draws. Expect ~8 min. The K=8 ARD
overparameterization caveat (latent params exceed observed cells when rows are
sparse) means sigma can be pulled low and in-sample R² is optimistic. The trace is
saved to `ppca_all_trace.nc` so the loadings cell below can be re-run without re-fitting.

In [8]:
import arviz as az

# Build + sample PPCA + ARD on the all-benchmarks matrix.
# 8 chains × 2000 draws.
ppca4 = build_ppca(prep4, K=K, noise="isotropic")

with ppca4:
    idata4 = pm.sample(
        draws=2000, tune=1000, chains=8, target_accept=0.9,
        random_seed=42, progressbar=False,
    )

# Persist so loadings cell below can be re-run without re-fitting.
idata4.to_netcdf("ppca_all_trace.nc")

n_div = int(idata4.sample_stats["diverging"].sum())
print(f"\ndivergences: {n_div} / {8*2000}")
print("\nidentified-quantity convergence (sigma):")
print(az.summary(idata4, var_names=["sigma"], round_to=4).to_string())

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (8 chains in 4 jobs)


NUTS: [Z, tau, W_z, sigma]


Sampling 8 chains for 1_000 tune and 2_000 draw iterations (8_000 + 16_000 draws total) took 810 seconds.


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



divergences: 0 / 16000

identified-quantity convergence (sigma):
         mean      sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk   ess_tail   r_hat
sigma  0.1982  0.0061  0.1863   0.2091     0.0003   0.0001  469.4046  2287.1107  1.0207


In [9]:
# Tau spectrum + in-sample reconstruction for the all-benchmarks fit.
post_all = idata4.posterior
S_full = post_all.sizes["chain"] * post_all.sizes["draw"]

tau_all = post_all["tau"].values.reshape(-1, K)
tau_sorted = np.sort(tau_all, axis=1)[:, ::-1]
tau_med = np.median(tau_sorted, axis=0)
tau_lo, tau_hi = np.percentile(tau_sorted, [5, 95], axis=0)

print(f"ARD tau spectrum, all benchmarks ({prep4.M}×{prep4.B}, K={K}):")
scale = 40 / tau_med[0]
for r in range(K):
    print(f"  axis {r+1}: {tau_med[r]:.3f}  [{tau_lo[r]:.3f}, {tau_hi[r]:.3f}]  "
          f"{'█' * int(round(tau_med[r] * scale))}")

thin = max(1, S_full // 500)
Z = post_all["Z"].values.reshape(-1, prep4.M, K)[::thin]
W = (post_all["W_z"] * post_all["tau"]).values.reshape(-1, prep4.B, K)[::thin]
mi, bi = prep4.model_idx, prep4.bench_idx
pred = (Z[:, mi, :] * W[:, bi, :]).sum(-1).mean(0)
resid = prep4.x - pred
rmse = float(np.sqrt((resid ** 2).mean()))
r2 = float(1 - (resid ** 2).sum() / (prep4.x ** 2).sum())
print(f"\nin-sample reconstruction: R² = {r2:.3f}, RMSE = {rmse:.3f} (logit scale)")
print("(optimistic — sparse rows overfit; treat as an upper bound)")

ARD tau spectrum, all benchmarks (291×69, K=8):
  axis 1: 0.885  [0.752, 1.055]  ████████████████████████████████████████
  axis 2: 0.520  [0.436, 0.632]  ████████████████████████
  axis 3: 0.237  [0.202, 0.290]  ███████████
  axis 4: 0.207  [0.181, 0.241]  █████████
  axis 5: 0.188  [0.164, 0.215]  ████████
  axis 6: 0.170  [0.146, 0.195]  ████████
  axis 7: 0.151  [0.126, 0.175]  ███████
  axis 8: 0.127  [0.095, 0.155]  ██████

in-sample reconstruction: R² = 0.990, RMSE = 0.115 (logit scale)
(optimistic — sparse rows overfit; treat as an upper bound)


# A.3 Prior overlay: is each axis informative vs the ARD prior?

**The test.** ARD's prior `tau_k ~ LogNormal(log 0.3, 1)` is what the model "expects"
*before* seeing data. If the data had zero signal, the posterior would just match this
prior. So we'd like to ask: at each rank, is the *observed* posterior tau clearly
larger than the *null* sorted prior (8 independent prior draws, sorted descending)?

**The wrinkle our run revealed.** Our prior has wide upper tails (sigma=1 in log
space). At K=8 the typical "biggest of 8 prior draws" is already ~1.2 — well above
our observed axis-1 median of 0.93. By this test, **all 8 axes show as "at prior"**
even though axes 1 and 2 are clearly doing real work (R²=0.99, clean
gap to the tail).

**The honest interpretation.** Two things at once:
- Our prior is too weakly informative for an "above prior" test to discriminate
  between real and shrunk axes. The prior gives so much room that "above the typical
  sorted prior" is a very high bar.
- The data IS informing the fit (observed posterior intervals are much narrower than
  the prior), but the discrimination test we'd hoped for doesn't land.

**What does discriminate, with the current prior.** The **gap structure within the
observed spectrum** — specifically the **2.1× jump from axis 2 (0.56) to axis 3
(0.26)**, which is much larger than every other consecutive ratio (~1.1–1.2×).
Axes 1 and 2 sit alone above; axes 3–8 form a tight smooth cluster, the prior-shrunk
tail — that's the 2-axis evidence under this prior.

**Caveat.** A cleaner statistical test would require a tighter ARD prior
(e.g., `LogNormal(log 0.1, 0.5)`) and a refit — but we shouldn't reverse-engineer
the prior just to make a diagnostic look good. Documented as a methodology note,
not retro-fixed.

In [10]:
# Prior overlay: sample 8 independent prior tau draws per "set," sort, summarize.
# Match the posterior sample size (8 chains × 5000 draws = 40,000).
rng = np.random.default_rng(42)
prior_tau = rng.lognormal(mean=np.log(0.3), sigma=1.0, size=(40_000, K))
prior_sorted = np.sort(prior_tau, axis=1)[:, ::-1]
prior_med = np.median(prior_sorted, axis=0)
prior_95  = np.percentile(prior_sorted, 95, axis=0)
prior_99  = np.percentile(prior_sorted, 99, axis=0)

# Observed posterior (sorted tau medians + 5% bounds) from the all-benchmarks PPCA fit.
post_all_for_overlay = idata4.posterior
tau_obs = post_all_for_overlay["tau"].values.reshape(-1, K)
tau_obs_sorted = np.sort(tau_obs, axis=1)[:, ::-1]
obs_med = np.median(tau_obs_sorted, axis=0)
obs_5   = np.percentile(tau_obs_sorted, 5, axis=0)

print(f"{'rank':4s} {'obs med':>8s} {'obs 5%':>8s} | "
      f"{'prior med':>10s} {'prior 95%':>10s} {'prior 99%':>10s} | verdict")
print("-" * 80)
for r in range(K):
    if obs_5[r] > prior_99[r]:
        v = "STRONG (obs 5% > prior 99%)"
    elif obs_5[r] > prior_95[r]:
        v = "above (obs 5% > prior 95%)"
    elif obs_med[r] > prior_95[r]:
        v = "weak (median above, bands overlap)"
    else:
        v = "AT PRIOR"
    print(f"{r+1:>4d} {obs_med[r]:>8.3f} {obs_5[r]:>8.3f} | "
          f"{prior_med[r]:>10.3f} {prior_95[r]:>10.3f} {prior_99[r]:>10.3f} | {v}")

# Gap analysis — the actual discriminator with this prior.
print("\nGap structure (consecutive ratios of observed sorted tau):")
for r in range(K - 1):
    ratio = obs_med[r] / obs_med[r+1]
    marker = "  ← biggest jump" if r == 1 else ""
    print(f"  axis {r+1} -> axis {r+2}: {obs_med[r]:.3f} / {obs_med[r+1]:.3f} = {ratio:.2f}x{marker}")

rank  obs med   obs 5% |  prior med  prior 95%  prior 99% | verdict
--------------------------------------------------------------------------------
   1    0.885    0.752 |      1.198      3.666      6.303 | AT PRIOR
   2    0.520    0.436 |      0.693      1.626      2.351 | AT PRIOR
   3    0.237    0.202 |      0.477      1.026      1.405 | AT PRIOR
   4    0.207    0.181 |      0.349      0.714      0.963 | AT PRIOR
   5    0.188    0.164 |      0.258      0.522      0.704 | AT PRIOR
   6    0.170    0.146 |      0.189      0.385      0.516 | AT PRIOR
   7    0.151    0.126 |      0.130      0.278      0.375 | AT PRIOR
   8    0.127    0.095 |      0.075      0.182      0.253 | AT PRIOR

Gap structure (consecutive ratios of observed sorted tau):
  axis 1 -> axis 2: 0.885 / 0.520 = 1.70x
  axis 2 -> axis 3: 0.520 / 0.237 = 2.20x  ← biggest jump
  axis 3 -> axis 4: 0.237 / 0.207 = 1.14x
  axis 4 -> axis 5: 0.207 / 0.188 = 1.11x
  axis 5 -> axis 6: 0.188 / 0.170 = 1.11x
  axis 6 -> a

# A.4 Loadings interpretation (axes 1 & 2 from PPCA)

The tau spectrum tells us *how many* axes the data uses; the loadings tell us
*what those axes are*. For each retained axis we list the top-+/− benchmarks.

**Two technical fixes are needed before posterior medians are meaningful:**

1. **Rank-tracking.** The 8 latent columns can rotate/permute across MCMC draws —
   "axis 3 in chain 1" might be "axis 7 in chain 2." We sort each draw's columns
   by `|tau|` descending so "rank-1" always means "biggest axis in this draw."
2. **Sign canonicalization.** Within a column the sign is unidentified
   (`Z·W = (-Z)·(-W)`). We canonicalize by power iteration: pick an initial
   reference loading vector, flip each draw to maximize its dot product with that
   reference, recompute the reference, repeat. Converges in 2–3 passes.

After these fixes, we report posterior median + 5–95% CI per benchmark, plus a
**sign% column** = fraction of draws agreeing with the median sign. High sign% (≥95%)
means the loading direction is stable across draws — the meaningful signal.
Loadings with sign% near 50% are sign-ambiguous and shouldn't be interpreted.

We use the **PPCA isotropic** trace as the only result here.

In [11]:
# Pull W and tau from the PPCA trace; do rank-tracking + sign-canonicalization.
W_all = idata4.posterior["W"].values
tau_all = idata4.posterior["tau"].values
C, D, B_, K_ = W_all.shape
W_all = W_all.reshape(C * D, B_, K_)
tau_all = tau_all.reshape(C * D, K_)

# 1. Sort each draw's columns by tau descending.
order = np.argsort(-tau_all, axis=1)
W_sorted = np.take_along_axis(W_all, order[:, None, :], axis=2)
tau_sorted = np.take_along_axis(tau_all, order, axis=1)

# 2. Sign-canonicalize each rank via power iteration.
for r in range(K_):
    col = W_sorted[:, :, r]
    ref = col[np.argmax(col.var(axis=1))].copy()
    for _ in range(3):
        signs = np.where(col @ ref < 0, -1.0, 1.0)
        col = col * signs[:, None]
        ref = col.mean(axis=0)
    W_sorted[:, :, r] = col

# 3. Report top-10 +/- loadings for axes 1 and 2.
bench_names_all = prep4.bench_names
for r in [0, 1]:
    med = np.median(W_sorted[:, :, r], axis=0)
    lo, hi = np.percentile(W_sorted[:, :, r], [5, 95], axis=0)
    agree = np.mean(np.sign(W_sorted[:, :, r]) == np.sign(med)[None, :], axis=0)
    tau_r = np.median(tau_sorted[:, r])

    print(f"\n{'='*78}")
    print(f"AXIS {r+1}   (tau median = {tau_r:.3f})")
    print(f"{'='*78}")
    print(f"  {'benchmark':40s} {'median':>8s} {'[5%, 95%]':>20s} {'sign%':>6s}")
    print(f"  {'-'*40} {'-'*8} {'-'*20} {'-'*6}")
    print(f"  --- top +loadings ---")
    for b in np.argsort(-med)[:10]:
        print(f"  {bench_names_all[b]:40s} {med[b]:>+8.3f} "
              f"[{lo[b]:>+7.3f}, {hi[b]:>+7.3f}] {agree[b]*100:>5.0f}%")
    print(f"  --- top -loadings ---")
    for b in np.argsort(med)[:10]:
        print(f"  {bench_names_all[b]:40s} {med[b]:>+8.3f} "
              f"[{lo[b]:>+7.3f}, {hi[b]:>+7.3f}] {agree[b]*100:>5.0f}%")


AXIS 1   (tau median = 0.885)
  benchmark                                  median            [5%, 95%]  sign%
  ---------------------------------------- -------- -------------------- ------
  --- top +loadings ---
  ARC-AGI-2                                  +2.528 [ +2.207,  +2.864]   100%
  FrontierMath Tier 4                        +2.235 [ +1.737,  +2.730]   100%
  OTIS Mock AIME 2024-2025                   +2.153 [ +1.798,  +2.441]   100%
  ARC-AGI                                    +1.904 [ +1.654,  +2.167]   100%
  FrontierMath                               +1.872 [ +1.648,  +2.098]   100%
  GSO-Bench                                  +1.648 [ +1.288,  +2.023]   100%
  Fiction.LiveBench                          +1.497 [ +0.662,  +2.458]   100%
  Aider Polyglot                             +1.434 [ +1.115,  +1.747]   100%
  Cybench                                    +1.418 [ +1.089,  +1.747]   100%
  EnigmaEval                                 +1.284 [ +1.100,  +1.481]   100%
  ---


AXIS 2   (tau median = 0.520)
  benchmark                                  median            [5%, 95%]  sign%
  ---------------------------------------- -------- -------------------- ------
  --- top +loadings ---
  FrontierMath Tier 4                        +1.081 [ +0.522,  +1.602]    99%
  GSO-Bench                                  +0.795 [ +0.365,  +1.216]   100%
  MCP Atlas                                  +0.689 [ +0.392,  +1.028]   100%
  SWE-Bench Pro (Private)                    +0.625 [ +0.287,  +0.959]   100%
  TerminalBench                              +0.584 [ +0.280,  +0.883]   100%
  PostTrainBench                             +0.496 [ +0.178,  +0.825]    99%
  APEX Agents                                +0.462 [ +0.149,  +0.766]    99%
  Chess Puzzles                              +0.361 [ +0.106,  +0.636]    99%
  SimpleQA Verified                          +0.311 [ +0.088,  +0.548]    99%
  SWE-Bench Verified                         +0.300 [ +0.139,  +0.462]   100%
  ---